# Lesson 4B: Gaussian Mixture Models - Practical

## Production GMM with Scikit-Learn

Apply GMM to real clustering problems with model selection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

In [ ]:
# Create overlapping clusters
X, y_true = make_blobs(n_samples=500, centers=4, n_features=2, 
                       cluster_std=1.2, random_state=42)

# Fit GMM with scikit-learn
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
gmm.fit(X)
y_pred = gmm.predict(X)
y_proba = gmm.predict_proba(X)

print(f"Log-likelihood: {gmm.score(X) * len(X):.2f}")
print(f"BIC: {gmm.bic(X):.2f}")
print(f"AIC: {gmm.aic(X):.2f}")
print(f"Converged: {gmm.converged_}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.6, edgecolors='k')
ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300, marker='*', 
          edgecolors='black', linewidths=2)
ax.set_title('GMM Clustering', fontweight='bold')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)

ax = axes[1]
max_proba = y_proba.max(axis=1)
scatter = ax.scatter(X[:, 0], X[:, 1], c=y_pred, s=max_proba*300,
                    cmap='viridis', alpha=0.6, edgecolors='k')
ax.set_title('Soft Assignments (size=confidence)', fontweight='bold')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ GMM applied!")

In [ ]:
# Model selection with BIC/AIC
n_components_range = range(1, 11)
bic_scores = []
aic_scores = []

for n_components in n_components_range:
    gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
    gmm.fit(X)
    bic_scores.append(gmm.bic(X))
    aic_scores.append(gmm.aic(X))

plt.figure(figsize=(10, 6))
plt.plot(n_components_range, bic_scores, 'bo-', label='BIC', linewidth=2, markersize=8)
plt.plot(n_components_range, aic_scores, 'rs-', label='AIC', linewidth=2, markersize=8)
plt.xlabel('Number of Components', fontweight='bold')
plt.ylabel('Information Criterion', fontweight='bold')
plt.title('Model Selection: BIC and AIC', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.axvline(x=4, color='green', linestyle='--', label='True K=4')
plt.show()

optimal_bic = n_components_range[np.argmin(bic_scores)]
optimal_aic = n_components_range[np.argmin(aic_scores)]
print(f"✅ Optimal components: BIC={optimal_bic}, AIC={optimal_aic}, True=4")

In [ ]:
# Compare covariance types
cov_types = ['spherical', 'diag', 'tied', 'full']
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, cov_type in enumerate(cov_types):
    gmm = GaussianMixture(n_components=4, covariance_type=cov_type, random_state=42)
    gmm.fit(X)
    y_pred = gmm.predict(X)
    
    ax = axes[idx // 2, idx % 2]
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis',
                        s=50, alpha=0.6, edgecolors='k')
    ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', s=300, marker='*',
              edgecolors='black', linewidths=2)
    ax.set_title(f'{cov_type.title()} Covariance\nBIC={gmm.bic(X):.0f}',
                fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Covariance Type Comparison', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print("✅ Compared covariance types!")